In [ ]:
from src import grid_load_simulator
grid_load_simulator.run_live_simulator()

  LIVE GRID LOAD SIMULATOR – Hoboken, NJ
  Writing to      : /home/jonathan/Power-Grid-Forecasting/data/generated/live_grid_load.csv
  Update interval : every 10 seconds
  History kept    : last 1000 samples
  Press Ctrl+C to stop.
[2026-08-16 20:26:11]  Load =  84.4 %   Temp =  34.6 °C   (Summer, Weekend)
[2026-08-16 20:26:21]  Load =  78.8 %   Temp =  32.5 °C   (Summer, Weekend)
[2026-08-16 20:26:31]  Load =  79.2 %   Temp =  33.3 °C   (Summer, Weekend)
[2026-08-16 20:26:41]  Load =  79.2 %   Temp =  31.2 °C   (Summer, Weekend)
[2026-08-16 20:26:51]  Load =  80.7 %   Temp =  31.8 °C   (Summer, Weekend)
[2026-08-16 20:27:01]  Load =  82.1 %   Temp =  33.7 °C   (Summer, Weekend)
[2026-08-16 20:27:11]  Load =  78.9 %   Temp =  33.0 °C   (Summer, Weekend)
[2026-08-16 20:27:21]  Load =  74.7 %   Temp =  28.4 °C   (Summer, Weekend)
[2026-08-16 20:27:31]  Load =  78.8 %   Temp =  32.0 °C   (Summer, Weekend)
[2026-08-16 20:27:41]  Load =  77.0 %   Temp =  30.2 °C   (Summer, Weekend)
[2026-08

In [ ]:
%%writefile data_loader.py
"""
Module responsible for loading, cleaning and merging all project datasets:
-Grid load data
-Weather Data
-Electrical outages
Module name: data_loader.py
"""
import pandas as pd
from datetime import datetime
import numpy as np

"""
Module responsible for loading, cleaning and merginf all project datasets:
-Grid load data
-Weather Data
-Electrical outages
Module name: data_loader.py
"""
import pandas as pd
from datetime import datetime
import numpy as np
import os

class DataLoader:
  """
  Handles reading CSV files, cleaning raw values, and merging datasets.
  """
  def __init__(self, grid_path: str, weather_path: str, outage_path: str):
        # return Exception if path doesnt exist
        if not os.path.exists(grid_path):
            raise FileNotFoundError(f"Grid data file not found: {grid_path}")
        if not os.path.exists(weather_path):
            raise FileNotFoundError(f"Weather data file not found: {weather_path}")
        if not os.path.exists(outage_path):
            raise FileNotFoundError(f"Outage data file not found: {outage_path}")   
            raise FileNotFoundError(f"Weather data file not found: {weather_path}")    

        self.grid_path = grid_path
        self.weather_path = weather_path
        self.outage_path = outage_path
  
  # Grid Load
  def load_grid_data(self):
    """
    Load the simulated grid data.
    """
    df = pd.read_csv(self.grid_path)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
    df = df.dropna(subset=['Timestamp'])
    return df

  # Weather Data Cleaning
  def _parse_noaa_numeric(self, val: str):
        """
        Extract the numeric portion for values like '+0001,1' or '99999,9'.      
        """
        if isinstance(val, str):
            try:
                raw = val.split(',')[0].replace('+', '')
                num = int(raw)
                if num > 9000:  # missing value indicator
                    return np.nan
                return num / 10.0  #TMP often stored as tenths of degrees C
            except:
                return np.nan
        return np.nan

  def load_weather_data(self) -> pd.DataFrame:
        """
        Load weather data and convert key fields.
        """
        df = pd.read_csv(self.weather_path)
        df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')
        df = df.dropna(subset=['DATE'])

        # Parse temperature
        if 'TMP' in df.columns:
            df['temperature_C'] = df['TMP'].apply(self._parse_noaa_numeric)
        else:
            df['temperature_C'] = np.nan

        # Parse dew point
        if 'DEW' in df.columns:
            df['dewpoint_C'] = df['DEW'].apply(self._parse_noaa_numeric)
        else:
            df['dewpoint_C'] = np.nan

        # Parse sea-level pressure
        if 'SLP (Sea Level Pressure)' in df.columns:
            df['pressure_hPa'] = df['SLP (Sea Level Pressure)'].apply(self._parse_noaa_numeric)
        else:
            df['pressure_hPa'] = np.nan

        return df[['DATE', 'temperature_C', 'dewpoint_C', 'pressure_hPa']]

  # Electrical Outage Data
  def load_outage_data(self):
    """
    Load historic electrical outage data for Hudson County.
    """
    df = pd.read_csv(self.outage_path)
    df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
    df = df.dropna(subset=['start_time'])
    return df
    
  # Merging  
  def merge_all(self):
    """
    Merge grid, weather and outage data.
    """
    grid = self.load_grid_data().sort_values('Timestamp')
    weather = self.load_weather_data().sort_values('DATE')
    merged = pd.merge_asof(
            grid,
            weather,
            left_on='Timestamp',
            right_on='DATE',
            direction='nearest'
        )

    merged.drop(columns=['DATE'], inplace=True)

    # Add simple outage indicator based on nearest outage events
    outages = self.load_outage_data()
    outages = outages.sort_values('start_time')

    merged['nearest_outage_customers'] = 0
    merged['nearest_outage_duration'] = 0.0

    idx = 0
    for i, row in merged.iterrows():
        ts = row['Timestamp']
        # Find closest outage in history
        while idx + 1 < len(outages) and outages.iloc[idx + 1]['start_time'] < ts:
             idx += 1
        outage = outages.iloc[idx]
        merged.at[i, 'nearest_outage_customers'] = outage['mean_customers']
        merged.at[i, 'nearest_outage_duration'] = outage['duration']

    return merged


Overwriting data_loader.py


In [4]:
%%writefile power_station.py
"""
Represents a Hudson County power station with load history and capacity.
Module name: power_station.py
"""
class PowerStation:
    def __init__(self, name: str, latitude: float, longitude: float, rated_capacity: float):
        self.name = name
        self.latitude = latitude
        self.longitude = longitude
        self.rated_capacity = rated_capacity
        self.load_history = []

    def add_load(self, load_percent: float):
        self.load_history.append(load_percent)

    def compute_average_load(self):
        if not self.load_history:
            return 0.0
        return sum(self.load_history) / len(self.load_history)

    def compute_peak_load(self):
        return max(self.load_history) if self.load_history else 0.0

    def __str__(self):
        return f"{self.name} @({self.latitude},{self.longitude}) capacity={self.rated_capacity}%"

    def __lt__(self, other):
        return self.rated_capacity < other.rated_capacity


Overwriting power_station.py


In [ ]:
%%writefile load_forecaster.py
"""
Implements simple short-term forecasting: rolling mean and seasonal weighting.
Module name: load_forecaster.py
"""
import pandas as pd

class LoadForecaster:
    def __init__(self, merged_df: pd.DataFrame):
        self.df = merged_df.copy()
        self.df = self.df.sort_values('Timestamp')

    def prepare_features(self):
        self.df['hour'] = self.df['Timestamp'].dt.hour
        self.df['weekday'] = self.df['Timestamp'].dt.weekday
        return self.df

    def forecast_next_hour(self):
        """
        Uses last N load values for rolling average forecasting.
        """
        recent = self.df['Load_Percent'].tail(6)  # last ~1 hour of 10-minute data
        return recent.mean()

    def forecast_next_24h(self):
        """
        Projects next 24 hours based on mean of each hour of day historically.
        """
        self.df['hour'] = self.df['Timestamp'].dt.hour
        hour_profile = self.df.groupby('hour')['Load_Percent'].mean()

        forecast = []
        for h in range(24):
            forecast.append(hour_profile.get(h, hour_profile.mean()))

        return pd.Series(forecast)

Overwriting load_forecaster.py


In [1]:
%%writefile outage_analyzer.py
"""
Computes outage frequency patterns, basic probabilities, and severity scores.
Module name: outage_analyzer.py
"""
import pandas as pd
import numpy as np

class OutageAnalyzer:
    def __init__(self, outage_df: pd.DataFrame):
        self.outages = outage_df.copy()
        self.outages['hour'] = self.outages['start_time'].dt.hour
        self.outages['month'] = self.outages['start_time'].dt.month

    def outages_by_season(self):
        return self.outages.groupby('month').size()

    def compute_outage_probability(self, timestamp):
        """
        Probability = (# of outages at same hour)/total outages
        """
        hour = timestamp.hour
        total = len(self.outages)
        occ = len(self.outages[self.outages['hour'] == hour])
        return occ / total if total > 0 else 0.0

    def recent_outages(self, timestamp, window_hours=6):
        """
        Returns outages in the last X hours.
        """
        start = timestamp - pd.Timedelta(hours=window_hours)
        return self.outages[(self.outages['start_time'] >= start) &
                            (self.outages['start_time'] <= timestamp)]

Overwriting outage_analyzer.py


In [ ]:
%%writefile risk_model.py
"""
Estimates risk from load, outage probability, and weather.
Module name: risk_model.py
"""
class RiskModel:
    def __init__(self, station_capacity: float):
        self.capacity = station_capacity

    def compute_risk_score(self, load_percent, outage_prob, temperature):
        """
        Weighted scoring model. Adjust freely.
        """
        load_factor = load_percent / self.capacity
        temp_factor = temperature / 40.0 if temperature is not None else 0.0
        risk = 0.6*load_factor + 0.25*outage_prob + 0.15*temp_factor
        return risk

    def classify(self, score):
        if score < 0.3:
            return "Low"
        elif score < 0.6:
            return "Moderate"
        elif score < 0.85:
            return "High"
        return "Critical"


Overwriting risk_model.py


In [ ]:
%%writefile utils.py
"""
Contains generator, decorators, and set operations.
Module name: utils.py
"""
import time
from functools import wraps

def runtime_logger(func):
    """
    Decorator logging runtime of any function.
    """
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        out = func(*args, **kwargs)
        print(f"{func.__name__} executed in {time.time() - start:.4f} sec")
        return out
    return wrapper

def hourly_load_generator(df):
    """
    Streams load entries row-by-row.
    """
    for _, row in df.iterrows():
        yield row

def unique_outage_days(outage_df):
    """
    Example set operation.
    """
    return set(outage_df['start_time'].dt.date)

Overwriting utils.py


In [ ]:
from data_loader import DataLoader
from power_station import PowerStation
from load_forecaster import LoadForecaster
from outage_analyzer import OutageAnalyzer
from risk_model import RiskModel

loader = DataLoader("../1341324.csv", "Weather data for Hudson station.csv", "Project electrical outages data.csv")
merged = loader.merge_all()

station = PowerStation("Hudson Substation", 40.728, -74.078, rated_capacity=85)
station.load_history = merged['Load_Percent'].tolist()

forecaster = LoadForecaster(merged)
forecast_hour = forecaster.forecast_next_hour()

outages = loader.load_outage_data()
out_analyzer = OutageAnalyzer(outages)
out_prob = out_analyzer.compute_outage_probability(merged['Timestamp'].iloc[-1])

risk_engine = RiskModel(station.rated_capacity)
risk_score = risk_engine.compute_risk_score(forecast_hour, out_prob, merged['temperature_C'].iloc[-1])
risk_level = risk_engine.classify(risk_score)

print("Next-hour forecast:", forecast_hour)
print("Outage probability:", out_prob)
print("Risk score:", risk_score, "=>", risk_level)

Next-hour forecast: 68.71209104379007
Outage probability: 0.03749043611323642
Risk score: 0.49214913404329785 => Moderate
